In [36]:
from promenade.models import *
from promenade.agents import build_graph, build_reranker_graph, build_filtring_agent
from langchain.tools import tool
from langchain.agents import create_agent

In [37]:
model = llm
reranker = RetreiveReranker(
    rerank_n=1000, 
    retrieve_n=10000, 
    rerank_model=RERANKER_MODEL)
WEB_TOOLS = WebTools(READER_URL)
DEV = False
SAMPLES_PATH = PROJECT_ROOT / "docs" / "samples"
base_url = "https://kosmo-museum.ru/" 
graph = build_graph(WEB_TOOLS, reranker, model, DEV=DEV, SAMPLES_PATH=SAMPLES_PATH)
reranker_graph = build_reranker_graph(llm=llm)
filtering_graph = build_filtring_agent(llm)

In [38]:
import json

@tool
def parse_and_insert_into_db(url: HttpUrl) -> str:
    """Parse web page and insert into DB.

    Args:
        url (HttpUrl): URL address of the page to parse.

    Returns:
        str: JSON string with keys:
            - status: "success" or "error"
            - message: human-readable description
            - details: additional context
    """
    try:
        state = asyncio.run(graph.ainvoke({"url": str(url), "subdocs": [], "results": []}))
        count  = sum(1 for r in state["results"] if r["ok"])
        result: ToolResult = {
            "status": "success", 
            "message": f"Sucssefully created {count} entities in the sql database and vesctor database.", 
            "details": {
                "count": count, 
                "result": state["results"]
            }
        }
        return json.dumps(result, ensure_ascii=False)

    except Exception as e:
        result: ToolResult = {
            "status": "error", 
            "message": str(e), 
            "details": {}
        }
        return json.dumps(result, ensure_ascii=False)
    

@tool
def extract_from_vector_stor(query: str) -> str:
    """Retrieve information about venues from Qdrant database.

    Args:
        query (str): User query in Russian. Passed exactly as-is to the database for vector search.

    Returns:
        str: JSON string with keys:
            - status: "success" or "error"
            - message: human-readable description
            - details: additional context with search results
    """
    try:
        state = asyncio.run(reranker_graph.ainvoke(
            {
                "input_query": query,
                "reranked_documents": None,
                "retrieved_count": None,
                "result": None
            }
        ))
        result: ToolResult = {
            "status": "success", 
            "message": f"Sucssefully retrieved information about {state["retrieved_count"]} places", 
            "details": {
                "result": state["result"]
            }
        }
        return json.dumps(result, ensure_ascii=False)
    except Exception as e:
        result: ToolResult = {
            "status": "error", 
            "message": str(e), 
            "details": {}
        }
        return json.dumps(result, ensure_ascii=False)
    

@tool
def filter_shedule_db(query: str) -> str:
    """Filter out available places in the schedule database.

    Args:
        query (str): User query in Russian. Passed exactly as-is to the database for filtered search.

    Returns:
        str: JSON string with keys:
            - status: "success" or "error"
            - message: human-readable description
            - details: search result sturctured as list of tuples (id, place_name, place_website_url)
    """
    print("in filter_shedule_db")
    try:
        state = filtering_graph.invoke(
            {
                "input_query": query,
                "reranked_documents": None,
                "retrieved_count": None,
                "result": None
            }
        )
        filtered_places = state["filtred_places"]
        places = "\n".join(f"{i+1}. {name} - {url}" for i, (_, name, url) in enumerate(filtered_places))
        result: ToolResult = {
            "status": "success", 
            "message": f"Sucssefully found information about {len(filtered_places)} places", 
            "details": {
                "result": places
            }
        }
        return json.dumps(result, ensure_ascii=False)
    except Exception as e:
        result: ToolResult = {
            "status": "error", 
            "message": str(e), 
            "details": {}
        }
        return json.dumps(result, ensure_ascii=False)

In [39]:
GENERAL_AGENT_SYSTEM_PROMPT = """
You are Promenade — an AI assistant for discovering cultural leisure activities in Moscow.

## Core Functionality
Promenade crawls venue pages (museums, exhibitions, concerts, festivals), extracts structured information,
persists data to SQL database and Qdrant vector store, and answers natural-language queries through retrieval-augmented generation.

## Available Modes (Agent Modes)
The agent supports multiple operating modes, selected based on user input or context:

### 1. Crawl & Store Mode (current)
When the user provides a URL to a venue page:
- Use `parse_and_insert_into_db` tool to crawl and store venue information
- Extracts schedule, ticket prices, exhibitions, and contact details
- Stores structured data in SQL database (`museum`, `schedule` tables)
- Creates vector embeddings in Qdrant for semantic search
- After getting a result of the tool only answer the user about quantity of entities created in database and their names.
- Do not suggest further interaction with user

### 2. Query & Retrieve Mode (current)
When the user asks natural-language questions about venues:
- Use `extract_from_vector_stor` tool to extract information from Qdrant database
- Use retrieval from Qdrant to find relevant venues
- Generate answers using LLM with retrieved context

### 3. Filter SQL scheduel base mode (current)
When a user wants to filter information from a database about places he can visit within a given time period:
- Filter venues by type (museum, concert, festival, etc.)
- Filter by schedule (open now, weekend, specific date)
- Filter by price range or special offers
- Pass the user's request directly to this tool without modification

## Guidelines
- In Crawl & Store mode, only use the provided URL (do not search for others)
- In Query mode, always cite sources from the retrieved data
- When uncertain, ask clarifying questions rather than making assumptions

## Output Format
- Use Russian for all responses unless user specifies otherwise
"""

agent = create_agent(
    model=model,
    tools=[parse_and_insert_into_db, extract_from_vector_stor, filter_shedule_db],
    system_prompt=GENERAL_AGENT_SYSTEM_PROMPT,
)

In [40]:
# agent_result = agent.invoke(
#     {"messages": [{"role": "user", "content": "Хочу как-нибудь сходить в https://pushkinmuseum.art/"}]},
# )
# print(agent_result)

In [41]:
agent_result = agent.invoke(
    {"messages": [{"role": "user", "content": "Куда я могу сходить в следующий вторник с 10 до 21"}]},
)
print(agent_result)

in filter_shedule_db
{'messages': [HumanMessage(content='Куда я могу сходить в следующий вторник с 10 до 21', additional_kwargs={}, response_metadata={}, id='afa70688-c036-43ff-9667-d3d49017022b'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 815, 'total_tokens': 953, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 800}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-6e1c397d-1de4-408a-b24a-a7ccade58de2', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2258-c470-7a82-a396-8f63ced2d7e1-0', tool_calls=[{'name': 'filter_shedule_db', 'args': {'query': 'Куда я могу сходить в следующий вторник с 10 до 21'}, 'id': 'chatcmpl-tool-ae9719385d614739', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 815, 'output_tokens': 138, 'total_tokens': 953, 

In [42]:
agent_result

{'messages': [HumanMessage(content='Куда я могу сходить в следующий вторник с 10 до 21', additional_kwargs={}, response_metadata={}, id='afa70688-c036-43ff-9667-d3d49017022b'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 815, 'total_tokens': 953, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 800}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'chatcmpl-6e1c397d-1de4-408a-b24a-a7ccade58de2', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e2258-c470-7a82-a396-8f63ced2d7e1-0', tool_calls=[{'name': 'filter_shedule_db', 'args': {'query': 'Куда я могу сходить в следующий вторник с 10 до 21'}, 'id': 'chatcmpl-tool-ae9719385d614739', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 815, 'output_tokens': 138, 'total_tokens': 953, 'input_token_detail

In [43]:
print(agent_result['messages'][-2].content)
print(agent_result['messages'][-1].content)

{"status": "success", "message": "Sucssefully found information about 3 places", "details": {"result": "1. Скалодром RedPoint (Москва) - http://redpoint.msk.ru/menu/ratesandhours/\n2. Главное здание - https://pushkinmuseum.art/\n3. Галерея искусства стран Европы и Америки - https://pushkinmuseum.art/"}}
В следующий вторник (с 10 : 00 до 21 : 00) в базе найдены три подходящих места:

1. **Скалодром RedPoint (Москва)** – часы работы и тарифы: <http://redpoint.msk.ru/menu/ratesandhours/>  
2. **Главное здание (Пушкинский музей)** – информация о работе: <https://pushkinmuseum.art/>  
3. **Галерея искусства стран Европы и Америки (Пушкинский музей)** – детали работы: <https://pushkinmuseum.art/>

Выбирайте, что вам ближе, и приятного времяпрепровождения!
